# What is Machine Learning?

**Topic:** Introduction to Machine Learning

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the difference between traditional programming and machine learning
- **Explain** how a machine learning model learns from data rather than following explicit rules
- **Identify** real-world applications and what type of data each model learns from

> **Tip:** Start by toggling the **Paradigm slider** between step 1 and step 2. Notice how the arrows reverse direction — that reversal is the entire idea of machine learning.

---
## How we got here

This notebook is the destination that your entire curriculum has been building toward. Every folder you worked through — tools, Python fundamentals, statistics, math, EDA, and preprocessing — was giving you the vocabulary and hands-on practice to understand what happens next.

You already know how to describe distributions (statistics/), measure spread and apply the chain rule (math/), and clean and encode real-world data (preprocessing/). Now we ask the question those skills were always leading toward: **what can a computer learn from data?**

---
## Why this matters for data science

Writing explicit rules for every possible situation is impractical. A spam filter built from hand-written `if/else` logic would need thousands of conditions and would still miss new patterns the moment spammers change tactics.

A machine learning model trained on millions of labeled examples discovers those patterns automatically. When the world changes, you retrain on new examples rather than manually update rules. That shift, from programming logic to learning from data, is what makes modern data science possible at scale.

---
## Try it yourself

In [2]:
out_diagram = Output()
out_apps    = Output()

step_slider = IntSlider(value=1, min=1, max=2, step=1,
    description="Paradigm:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="380px"))

app_dd = Dropdown(
    options=["Spam detection", "Movie recommendation",
             "Image recognition", "Loan approval", "Medical diagnosis"],
    value="Spam detection",
    description="Application:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="380px"))

APP_INFO = {
    "Spam detection":       ("Word patterns in email text",
                             "Millions of labeled emails (spam / not spam)",
                             "Whether a new email is spam"),
    "Movie recommendation": ("Which movies similar users enjoyed",
                             "Watch history and star ratings from millions of users",
                             "Movies you are likely to enjoy"),
    "Image recognition":    ("Edges, shapes, and textures in pixels",
                             "Millions of labeled images",
                             "What objects appear in a new image"),
    "Loan approval":        ("Historical repayment patterns by applicant profile",
                             "Historical loan records with outcomes",
                             "Likelihood a new applicant will repay"),
    "Medical diagnosis":    ("Symptom combinations linked to confirmed diagnoses",
                             "Patient records with confirmed diagnoses",
                             "Most likely condition for a new patient"),
}

def render_diagram(step):
    if step == 1:
        labels = ["Rules", "Data", "Output"]
        colors = [PALETTE["secondary"], PALETTE["muted"], PALETTE["primary"]]
        title  = "Traditional programming: Rules + Data → Output"
    else:
        labels = ["Data", "Labels", "Model"]
        colors = [PALETTE["muted"], PALETTE["muted"], PALETTE["primary"]]
        title  = "Machine learning: Data + Labels → Model (learned rules)"
    xs = [0.15, 0.50, 0.85]
    traces = []
    for x, lbl, col in zip(xs, labels, colors):
        traces.append(go.Scatter(x=[x], y=[0.5], mode="markers+text",
            marker=dict(size=90, color=col, opacity=0.85),
            text=[lbl], textposition="middle center",
            textfont=dict(color="white", size=13, family=FONT["family"]),
            showlegend=False, hoverinfo="skip"))
    for i in range(2):
        traces.append(go.Scatter(
            x=[(xs[i]+xs[i+1])/2], y=[0.5],
            mode="text", text=["→"],
            textfont=dict(size=24, color=PALETTE["muted"]),
            showlegend=False, hoverinfo="skip"))
    layout = base_layout(title=title)
    layout.update(xaxis=dict(visible=False, range=[0,1]),
                  yaxis=dict(visible=False, range=[0,1]), height=220)
    with out_diagram:
        clear_output(wait=True)
        go.Figure(data=traces, layout=layout).show()

def render_app(app):
    learns, data, predicts = APP_INFO[app]
    text = ("<b>What the model learns:</b> " + learns + "<br>" +
            "<b>Training data:</b> " + data + "<br>" +
            "<b>What it predicts:</b> " + predicts)
    trace = go.Scatter(x=[0], y=[0], mode="text", text=[text],
        textfont=dict(size=13, family=FONT["family"]),
        hoverinfo="skip", showlegend=False)
    layout = base_layout(title="ML in action: " + app)
    layout.update(xaxis=dict(visible=False, range=[-1,1]),
                  yaxis=dict(visible=False, range=[-1,1]), height=180)
    with out_apps:
        clear_output(wait=True)
        go.Figure(data=[trace], layout=layout).show()

def on_step(change):  render_diagram(step_slider.value)
def on_app(change):   render_app(app_dd.value)

step_slider.observe(on_step, names="value")
app_dd.observe(on_app,       names="value")

display(VBox([step_slider, out_diagram, app_dd, out_apps]))
render_diagram(1)
render_app("Spam detection")

---
## What's happening?

Traditional programming and machine learning solve problems in opposite directions.

**Traditional programming:** A developer writes explicit rules. The computer applies those rules to new data and produces output. The rules come from human knowledge written before the fact.

**Machine learning:** The direction is reversed. You provide data (thousands of emails) and correct answers (which ones were spam). The algorithm finds the rules itself by identifying patterns that separate spam from legitimate mail.

The model is those learned patterns, stored as numbers. There is no human-readable rulebook, but the patterns generalize to new emails the model has never seen before.

**Why this matters:** You do not need to anticipate every rule in advance. The model discovers patterns that humans might never notice, and it handles situations no rule-writer predicted.

---
## Real-world example: Spam detection

A rule-based filter might check for phrases like "free money." Spammers adapt: they write "fr33 m0n3y" or embed text in images. Every evasion requires a manual rule update.

A machine learning model trained on millions of labeled emails learns the underlying structure of spam, not just surface-level keywords. The chart below shows how two simple features already separate most spam from legitimate email, even without a single hand-written rule.

In [3]:
np.random.seed(42)
n = 200
contains_free = np.random.choice([0, 1], n, p=[0.4, 0.6])
has_url       = np.random.choice([0, 1], n, p=[0.5, 0.5])
spam_prob     = 0.1 + 0.5*contains_free + 0.3*has_url
is_spam       = (np.random.random(n) < spam_prob).astype(int)

traces = [
    go.Scatter(
        x=contains_free[is_spam==0] + np.random.normal(0, 0.05, (is_spam==0).sum()),
        y=has_url[is_spam==0]       + np.random.normal(0, 0.05, (is_spam==0).sum()),
        mode="markers",
        marker=dict(color=PALETTE["accent"], size=8, opacity=0.65),
        name="Legitimate"),
    go.Scatter(
        x=contains_free[is_spam==1] + np.random.normal(0, 0.05, (is_spam==1).sum()),
        y=has_url[is_spam==1]       + np.random.normal(0, 0.05, (is_spam==1).sum()),
        mode="markers",
        marker=dict(color=PALETTE["secondary"], size=8, opacity=0.65),
        name="Spam"),
]
layout = base_layout(
    title="Spam detection: ML learns the boundary automatically",
    xaxis_title="Contains suspicious words (0=No, 1=Yes)",
    yaxis_title="Contains URLs (0=No, 1=Yes)")
layout.update(xaxis=dict(range=[-0.3,1.3]), yaxis=dict(range=[-0.3,1.3]))
go.Figure(data=traces, layout=layout).show()

### ML applications at a glance

| Application | What it learns | Training data | What it predicts |
|---|---|---|---|
| Spam detection | Word and link patterns | Labeled emails | Spam or not spam |
| Movie recommendations | Viewer preference patterns | Ratings from millions of users | Movies you will enjoy |
| Medical diagnosis | Symptom-to-diagnosis patterns | Patient records | Likely condition |
| Fraud detection | Unusual transaction patterns | Historical transactions with labels | Fraudulent or legitimate |
| Image recognition | Visual patterns (edges, shapes, textures) | Millions of labeled images | Object type in new image |

---
## Key takeaway

> **Machine learning flips the direction of programming: instead of writing rules to produce outputs, you provide labeled examples and let the algorithm discover the rules from data.**

---
*Next up: Supervised vs Unsupervised Learning — two fundamentally different ways a machine can learn*